# Split catalog into training and eval sets

This script should split the catalog csv in the root ShearNet directory into 3/5 and 2/5 fits. The 3/5 will be used for training and the 2/5 for evaluation of my network. 

The reason we have to do this is because we don't want to evaluate on the same data the we train on.

We also have to visualize the distribution of data across the four main parameters shearnet trains on so we know the two eval and train sets have the same distribution.

In [1]:
from astropy.table import Table
import galsim
import numpy as np

cosmos_cat_fname = "/home/adfield/ShearNet/cosmos15_superbit2023_phot_shapes_with_sigma.csv"
cosmos_cat = Table.read(cosmos_cat_fname, format="csv")

q = cosmos_cat['c10_sersic_fit_q']
phi = cosmos_cat['c10_sersic_fit_phi']

g1_list = []
g2_list = []
for qi, phi_i in zip(q, phi):
    if qi > 1.0:
        qi = 1.0 / qi
    s = galsim.Shear(q=float(qi), beta=float(phi_i) * galsim.radians)
    g1_list.append(s.g1)
    g2_list.append(s.g2)

g1_list = np.array(g1_list)
g2_list = np.array(g2_list)

half_light_radius = cosmos_cat['c10_sersic_fit_hlr'] * 0.03 * np.sqrt(q)
half_light_radius = np.minimum(half_light_radius, 1.0)
min_hlr = 1e-6
hlr_list = np.where(
    np.isfinite(half_light_radius) & (half_light_radius > 0),
    half_light_radius,
    min_hlr
)

flux_list = cosmos_cat['crates_b'] * 300 / 0.343